In [53]:
import pandas as pd
import numpy as np
import plotly as px

In [54]:
profile = pd.read_excel("Datasets/Client Profiles.xlsx")
survey = pd.read_excel("Datasets/Client Survey.xlsx")
cogs = pd.read_excel("Datasets/Revenue and COGS.xlsx")

In [55]:
profile['COUNTRY'] = np.where(profile['SG-BASED'].str.lower().isin(['yes', 'y']), 'Singapore', profile['COUNTRY OF OVERSEAS CLIENT'])

In [56]:
profile['STAFF STRENGTH'] = profile['STAFF STRENGTH'].replace({'More than 200': "> 200", 'Less than 200': "50 ~ 200", 'Less than 49': "1 ~ 49"})

In [57]:
cogs

,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER
0,G1-001,2021,148180,17321,25562,41201
1,G1-001,2022,197313,23604,32875,50085
2,G1-001,2023,349899,51332,71695,89089
3,G1-001,2024,268821,40969,56452,76545
4,G1-001,2025,138115,14516,20889,28968
...,...,...,...,...,...,...
418,P3-027,2025,172191,17127,35088,52215
419,P3-028,2025,220048,15847,45103,91424
420,P3-029,2025,279839,22351,45673,70801
421,P3-030,2025,146829,14505,30602,43338


In [58]:
# Merge the three datasets into a single dataframe on Client ID
client_id_candidates = ["Client ID", "CLIENT ID", "ClientID", "CLIENTID", "client_id", "Client_Id", "Client Id"]


def _resolve_client_id(df, df_name):
    for candidate in client_id_candidates:
        if candidate in df.columns:
            return candidate
    raise KeyError(f"No client ID column found in {df_name}. Checked: {client_id_candidates}")

profile_id = _resolve_client_id(profile, "profile")
survey_id = _resolve_client_id(survey, "survey")
cogs_id = _resolve_client_id(cogs, "cogs")




In [59]:
survey_duplicates = survey[survey.duplicated(keep=False)].sort_values(
    ["CLIENT ID", "YEAR"]
)

cogs_duplicates = cogs[cogs.duplicated(keep=False)].sort_values(
    ["CLIENT ID", "YEAR"]
)

display(survey_duplicates)
display(cogs_duplicates)

survey_clean = survey.drop_duplicates().reset_index(drop=True)
cogs_clean = cogs.drop_duplicates().reset_index(drop=True)

,YEAR,CLIENT ID,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING
0,2021,G1-001,3,4,4,5,8
3,2021,G1-001,3,4,4,5,8
33,2022,G1-001,5,5,3,4,9
37,2022,G1-001,5,5,3,4,9
80,2023,G1-001,5,4,3,4,7
88,2023,G1-001,5,4,3,4,7
164,2024,G1-001,5,5,3,4,8
179,2024,G1-001,5,5,3,4,8
1,2021,G1-002,3,4,4,2,7
4,2021,G1-002,3,4,4,2,7


,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER
27,G1-010,2023,302802,38429,58676,109502
28,G1-010,2023,302802,38429,58676,109502
29,G1-010,2024,336405,49777,65314,76727
30,G1-010,2024,336405,49777,65314,76727
31,G1-010,2025,737373,89810,134715,253187
32,G1-010,2025,737373,89810,134715,253187
57,N1-001,2021,62351,11844,16111,21089
58,N1-001,2021,62351,11844,16111,21089
72,N1-004,2023,28123,5525,7284,6309
73,N1-004,2023,28123,5525,7284,6309


In [60]:
invalid_survey = survey_clean[
    ~survey_clean["CLIENT ID"].isin(profile["CLIENT ID"])
]

invalid_cogs = cogs_clean[
    ~cogs_clean["CLIENT ID"].isin(profile["CLIENT ID"])
]

display(invalid_survey)
display(invalid_cogs)

,YEAR,CLIENT ID,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING
14,2021,P1-010,4,3,3,5,7
47,2022,P1-010,5,4,3,4,8
97,2023,P1-010,4,3,3,4,8
104,2023,P1-018,5,3,5,3,9
185,2024,P1-010,5,4,5,4,9
191,2024,P1-018,4,3,3,4,7
208,2024,P1-035,5,5,5,4,9
308,2025,P1-010,4,5,5,4,9
313,2025,P1-018,3,3,4,2,5
329,2025,P1-035,4,4,4,5,8


,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER
133,P1-010,2021,138674,23596,39801,31367
134,P1-010,2022,608823,83888,126808,114452
135,P1-010,2023,199495,32293,54252,42627
136,P1-010,2024,725356,79963,138332,119101
137,P1-010,2025,378450,53701,92626,74211
162,P1-018,2023,675484,93067,180793,137033
163,P1-018,2024,828849,97622,201052,148108
164,P1-018,2025,401230,58872,105479,80949
211,P1-035,2024,270266,33712,62853,46282
212,P1-035,2025,748979,111263,147531,151014


In [61]:
print("Survey rows:", len(survey_clean))
print("COGS rows:", len(cogs_clean))

Survey rows: 402
COGS rows: 402


In [62]:
survey_clean = survey_clean[
    survey_clean["CLIENT ID"].isin(profile["CLIENT ID"])
].reset_index(drop=True)

cogs_clean = cogs_clean[
    cogs_clean["CLIENT ID"].isin(profile["CLIENT ID"])
].reset_index(drop=True)

In [63]:
print("Survey rows:", len(survey_clean))
print("COGS rows:", len(cogs_clean))

Survey rows: 392
COGS rows: 392


In [64]:
df = (
    profile.rename(columns={profile_id: "Client ID"})
    .merge(survey.rename(columns={survey_id: "Client ID"}), on="Client ID", how="left", suffixes=('', '_survey'))
    .merge(cogs.rename(columns={cogs_id: "Client ID"}), on="Client ID", how="left", suffixes=('', '_cogs'))
)

df.head()

,Client ID,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,COUNTRY,YEAR,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,YEAR_cogs,REVENUE,HARDWARE,SOFTWARE,MANPOWER
0,G1-001,Govt,Yes,NaN,,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2021.0,148180.0,17321.0,25562.0,41201.0
1,G1-001,Govt,Yes,NaN,,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2022.0,197313.0,23604.0,32875.0,50085.0
2,G1-001,Govt,Yes,NaN,,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2023.0,349899.0,51332.0,71695.0,89089.0
3,G1-001,Govt,Yes,NaN,,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2024.0,268821.0,40969.0,56452.0,76545.0
4,G1-001,Govt,Yes,NaN,,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2025.0,138115.0,14516.0,20889.0,28968.0


In [65]:
df

,Client ID,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,COUNTRY,YEAR,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,YEAR_cogs,REVENUE,HARDWARE,SOFTWARE,MANPOWER
0,G1-001,Govt,Yes,NaN,,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2021.0,148180.0,17321.0,25562.0,41201.0
1,G1-001,Govt,Yes,NaN,,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2022.0,197313.0,23604.0,32875.0,50085.0
2,G1-001,Govt,Yes,NaN,,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2023.0,349899.0,51332.0,71695.0,89089.0
3,G1-001,Govt,Yes,NaN,,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2024.0,268821.0,40969.0,56452.0,76545.0
4,G1-001,Govt,Yes,NaN,,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2025.0,138115.0,14516.0,20889.0,28968.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1532,P3-027,Private,,Y,Malaysian,2025-01-04 00:00:00,> 200,Info Tech,Malaysian,2025.0,4.0,5.0,5.0,3.0,9.0,2025.0,172191.0,17127.0,35088.0,52215.0
1533,P3-028,Private,NaN,Y,India,2025-01-14 00:00:00,50 ~ 200,Info Tech,India,2025.0,4.0,5.0,5.0,5.0,9.0,2025.0,220048.0,15847.0,45103.0,91424.0
1534,P3-029,Private,,Y,Malaysia,2025-04-04 00:00:00,> 200,Healthcare,Malaysia,2025.0,5.0,2.0,4.0,5.0,9.0,2025.0,279839.0,22351.0,45673.0,70801.0
1535,P3-030,Private,NaN,Y,Indonesia,2025-05-04 00:00:00,50 ~ 200,Healthcare,Indonesia,2025.0,3.0,3.0,3.0,3.0,6.0,2025.0,146829.0,14505.0,30602.0,43338.0


In [66]:
profile

,CLIENT ID,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,COUNTRY
0,G1-001,Govt,Yes,NaN,,2020-01-19 00:00:00,> 200,Finance,Singapore
1,G1-002,Govt,Yes,NaN,,2020-05-19 00:00:00,> 200,Security,Singapore
2,G1-003,Govt,Yes,NaN,,2021-07-18 00:00:00,> 200,Transportation,Singapore
3,G1-005,Govt,Y,NaN,NaN,2022-02-11 00:00:00,> 200,Security,Singapore
4,G1-006,Govt,Y,NaN,NaN,2022-03-18 00:00:00,50 ~ 200,Legal,Singapore
...,...,...,...,...,...,...,...,...,...
143,P3-027,Private,,Y,Malaysian,2025-01-04 00:00:00,> 200,Info Tech,Malaysian
144,P3-028,Private,NaN,Y,India,2025-01-14 00:00:00,50 ~ 200,Info Tech,India
145,P3-029,Private,,Y,Malaysia,2025-04-04 00:00:00,> 200,Healthcare,Malaysia
146,P3-030,Private,NaN,Y,Indonesia,2025-05-04 00:00:00,50 ~ 200,Healthcare,Indonesia


In [67]:
cogs

,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER
0,G1-001,2021,148180,17321,25562,41201
1,G1-001,2022,197313,23604,32875,50085
2,G1-001,2023,349899,51332,71695,89089
3,G1-001,2024,268821,40969,56452,76545
4,G1-001,2025,138115,14516,20889,28968
...,...,...,...,...,...,...
418,P3-027,2025,172191,17127,35088,52215
419,P3-028,2025,220048,15847,45103,91424
420,P3-029,2025,279839,22351,45673,70801
421,P3-030,2025,146829,14505,30602,43338


In [68]:
listcol = ["SG-BASED", "COUNTRY OF OVERSEAS CLIENT", "OVERSEAS"]
df.drop(columns=listcol, axis=1, inplace=True)


In [69]:
df = df.dropna()
df

,Client ID,TYPE,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,COUNTRY,YEAR,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,YEAR_cogs,REVENUE,HARDWARE,SOFTWARE,MANPOWER
0,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2021.0,148180.0,17321.0,25562.0,41201.0
1,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2022.0,197313.0,23604.0,32875.0,50085.0
2,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2023.0,349899.0,51332.0,71695.0,89089.0
3,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2024.0,268821.0,40969.0,56452.0,76545.0
4,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2025.0,138115.0,14516.0,20889.0,28968.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1532,P3-027,Private,2025-01-04 00:00:00,> 200,Info Tech,Malaysian,2025.0,4.0,5.0,5.0,3.0,9.0,2025.0,172191.0,17127.0,35088.0,52215.0
1533,P3-028,Private,2025-01-14 00:00:00,50 ~ 200,Info Tech,India,2025.0,4.0,5.0,5.0,5.0,9.0,2025.0,220048.0,15847.0,45103.0,91424.0
1534,P3-029,Private,2025-04-04 00:00:00,> 200,Healthcare,Malaysia,2025.0,5.0,2.0,4.0,5.0,9.0,2025.0,279839.0,22351.0,45673.0,70801.0
1535,P3-030,Private,2025-05-04 00:00:00,50 ~ 200,Healthcare,Indonesia,2025.0,3.0,3.0,3.0,3.0,6.0,2025.0,146829.0,14505.0,30602.0,43338.0


In [70]:
print("Merged duplicates:", df.duplicated().sum())
df[df.duplicated(keep=False)]

Merged duplicates: 179


,Client ID,TYPE,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,COUNTRY,YEAR,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,YEAR_cogs,REVENUE,HARDWARE,SOFTWARE,MANPOWER
0,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2021.0,148180.0,17321.0,25562.0,41201.0
1,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2022.0,197313.0,23604.0,32875.0,50085.0
2,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2023.0,349899.0,51332.0,71695.0,89089.0
3,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2024.0,268821.0,40969.0,56452.0,76545.0
4,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2021.0,3.0,4.0,4.0,5.0,8.0,2025.0,138115.0,14516.0,20889.0,28968.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1471,P3-017,Private,2023-03-02 00:00:00,> 200,Manufacturing,China,2025.0,3.0,3.0,3.0,3.0,5.0,2023.0,153406.0,9296.0,20798.0,36782.0
1472,P3-017,Private,2023-03-02 00:00:00,> 200,Manufacturing,China,2025.0,3.0,3.0,3.0,3.0,5.0,2024.0,335347.0,20414.0,50192.0,82885.0
1473,P3-017,Private,2023-03-02 00:00:00,> 200,Manufacturing,China,2025.0,3.0,3.0,3.0,3.0,5.0,2024.0,335347.0,20414.0,50192.0,82885.0
1474,P3-017,Private,2023-03-02 00:00:00,> 200,Manufacturing,China,2025.0,3.0,3.0,3.0,3.0,5.0,2025.0,212601.0,15319.0,29686.0,41543.0


In [71]:
df.to_excel('merged.xlsx', index=False)